In [1]:
from google.colab import files
uploaded = files.upload()  # Choose the ZIP file from your computer


Saving dataset.zip to dataset.zip


In [2]:
import os

folder = "/content/dataset"
if os.path.exists(folder):
    !rm -r {folder}
    print(f"{folder} deleted.")
else:
    print("Folder does not exist.")


Folder does not exist.


In [3]:
%pip install python-Levenshtein

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.9/159.9 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 66.0 MB/s eta 0:00:00


In [4]:
import zipfile

zip_path = "/content/dataset.zip"  # Replace with your ZIP filename
extract_path = "/content/dataset"         # Folder where you want to extract

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)


In [6]:
import os
import re
import unicodedata
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import sentencepiece as spm
from torch.nn.utils.rnn import pad_sequence
from tqdm import tqdm
import math
import copy
import Levenshtein
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
smooth_fn = SmoothingFunction().method1

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ============================
# Early Stopping
# ============================
class EarlyStopping:
    def __init__(self, patience=5, mode="max"):
        self.patience = patience
        self.mode = mode
        self.best_score = None
        self.counter = 0
        self.best_model_enc = None
        self.best_model_dec = None
        self.early_stop = False

    def __call__(self, score, encoder, decoder):
        if self.best_score is None or \
           (self.mode=="max" and score>self.best_score) or \
           (self.mode=="min" and score<self.best_score):
            self.best_score = score
            self.best_model_enc = copy.deepcopy(encoder.state_dict())
            self.best_model_dec = copy.deepcopy(decoder.state_dict())
            self.counter = 0
        else:
            self.counter += 1
            print(f"No improvement. Patience: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True

    def load_best_model(self, encoder, decoder):
        encoder.load_state_dict(self.best_model_enc)
        decoder.load_state_dict(self.best_model_dec)

# ============================
# Preprocessing
# ============================
def preprocess_urdu(text):
    text = unicodedata.normalize("NFC", text)
    text = re.sub(r"[\u064B-\u065F\u0670]", "", text)
    text = text.replace("۔", ".").replace("؟", "?").replace("،", ",")
    text = re.sub(r"\s+", " ", text).strip()
    return text

def preprocess_roman(text):
    text = text.lower()
    text = re.sub(r"(.)\1{2,}", r"\1\1", text)
    text = re.sub(r"[^a-z0-9 ,?.!]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


# ============================
# Urdu → Roman Transliteration Function
# ============================
def urdu_to_roman(text):
    # Base character mapping
    text = text.replace("ي", "ی")
    mapping = {
        "ا":"a", "آ":"aa", "ب":"b", "پ":"p", "ت":"t", "ٹ":"ṭ",
        "ث":"s", "ج":"j", "چ":"ch", "ح":"h", "خ":"kh",
        "د":"d", "ڈ":"ḍ", "ذ":"z", "ر":"r", "ڑ":"ṛ",
        "ز":"z", "ژ":"zh", "س":"s", "ش":"sh", "ص":"s",
        "ض":"z", "ط":"t", "ظ":"z", "ع":"'", "غ":"gh",
        "ف":"f", "ق":"q", "ک":"k", "گ":"g", "ل":"l",
        "م":"m", "ن":"n", "و":"w", "ہ":"h", "ء":"'",
        "ی":"y", "ے":"e", "ں":"n", "ئ":"y", "ھ":"h"
    }

    # Short vowel guesses (zabar, zer, pesh approximation)
    vowel_rules = {
        "ب": "a", "ت": "a", "د": "a", "ک": "a", "ل": "a",
        "م": "a", "ن": "a", "ف": "a", "س": "a", "ش": "i",
        "ج": "a", "چ": "a", "ر": "a"
    }

    roman = ""
    for i, ch in enumerate(text):
        r = mapping.get(ch, ch)
        roman += r

        # Add an approximate vowel if consonant followed by consonant
        if ch in vowel_rules:
            if i+1 < len(text):
                nxt = text[i+1]
                if nxt in mapping and mapping[nxt] not in "aeiouy":
                    roman += vowel_rules[ch]

    return roman.strip()


# ============================
# Load Dataset Lines (Corrected)
# ============================
def load_dataset_lines(dataset_dir):
    urdu_texts = []
    roman_texts = []
    raw_urdu_count = 0
    raw_roman_count = 0

    for poet in os.listdir(dataset_dir):
        poet_path = os.path.join(dataset_dir, poet)
        if not os.path.isdir(poet_path):
            continue
        ur_path = os.path.join(poet_path, "ur")
        en_path = os.path.join(poet_path, "en")
        if not os.path.exists(ur_path):
            print(f"Skipping poet {poet} due to missing 'ur' directory.")
            continue

        for fname in os.listdir(ur_path):
            ur_file = os.path.join(ur_path, fname)
            en_file = os.path.join(en_path, fname)

            # Load Urdu
            with open(ur_file, "r", encoding="utf-8") as f_ur:
                ur_lines = [preprocess_urdu(line.strip()) for line in f_ur if line.strip()]
                raw_urdu_count += len(ur_lines)

            # Load Roman (if exists, else empty)
            en_lines = []
            if os.path.exists(en_file):
                with open(en_file, "r", encoding="utf-8") as f_en:
                    en_lines = [preprocess_roman(line.strip()) for line in f_en if line.strip()]
                    raw_roman_count += len(en_lines)

            # Alignment check
            if len(ur_lines) != len(en_lines):
                print(f" Line count mismatch in {fname}. Urdu={len(ur_lines)}, Roman={len(en_lines)}")

            # Case 1: Urdu has more → generate missing Roman
            if len(ur_lines) > len(en_lines):
                new_lines = []
                # Start from the *next* Urdu line
                for i in range(len(en_lines), len(ur_lines)):
                    roman_line = urdu_to_roman(ur_lines[i])
                    tagged_line = "[AUTO] " + roman_line
                    en_lines.append(tagged_line)
                    new_lines.append(tagged_line)

                # Append missing lines into file
                os.makedirs(en_path, exist_ok=True)
                with open(en_file, "a", encoding="utf-8") as f_en:
                    for line in new_lines:
                        f_en.write(line + "\n")
                print(f" Added {len(new_lines)} Roman lines to {en_file}")

            # Case 2: Roman has more → trim extras
            elif len(en_lines) > len(ur_lines):
                en_lines = en_lines[:len(ur_lines)]
                with open(en_file, "w", encoding="utf-8") as f_en:
                    for line in en_lines:
                        f_en.write(line + "\n")
                print(f" Trimmed extra Roman lines in {en_file}")

            # Final alignment (both now equal length)
            for i in range(len(ur_lines)):
                urdu_texts.append(ur_lines[i].strip())
                roman_texts.append(en_lines[i].strip())

    # Print summary ONCE at the end
    print(f"Raw Urdu lines (before alignment): {raw_urdu_count}")
    print(f"Raw Roman lines (before alignment): {raw_roman_count}")
    print(f"Aligned Urdu lines (after alignment): {len(urdu_texts)}")
    print(f"Aligned Roman lines (after alignment): {len(roman_texts)}")

    return urdu_texts, roman_texts


# ============================
# Example Usage
# ============================
dataset_dir = "/content/dataset/dataset"
urdu_texts, roman_texts = load_dataset_lines(dataset_dir)



# ============================
# Train SentencePiece
# ============================
with open("urdu_corpus.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(urdu_texts))
with open("roman_corpus.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(roman_texts))

spm.SentencePieceTrainer.Train("--input=urdu_corpus.txt --model_prefix=sp_urdu --vocab_size=6000")
spm.SentencePieceTrainer.Train("--input=roman_corpus.txt --model_prefix=sp_roman --vocab_size=6000")

sp_urdu = spm.SentencePieceProcessor(model_file="sp_urdu.model")
sp_roman = spm.SentencePieceProcessor(model_file="sp_roman.model")

src_vocab_size = sp_urdu.get_piece_size()
tgt_vocab_size = sp_roman.get_piece_size()
print(f"Urdu vocab: {src_vocab_size}, Roman vocab: {tgt_vocab_size}")

# ============================
# Dataset & DataLoader
# ============================
class Seq2SeqDataset(Dataset):
    def __init__(self, src_texts, tgt_texts, sp_src, sp_tgt, max_len=50):
        self.src = [sp_src.EncodeAsIds(s) for s in src_texts]
        self.tgt = [sp_tgt.EncodeAsIds(s) for s in tgt_texts]
        self.max_len = max_len
        self.sp_src = sp_src
        self.sp_tgt = sp_tgt

    def __len__(self):
        return len(self.src)

    def __getitem__(self, idx):
        src_ids = self.src[idx][:self.max_len]
        tgt_ids = [self.sp_tgt.bos_id()] + self.tgt[idx][:self.max_len] + [self.sp_tgt.eos_id()]
        return torch.tensor(src_ids), torch.tensor(tgt_ids)

def collate_fn(batch):
    src_batch, tgt_batch = zip(*batch)
    src_batch = pad_sequence(src_batch, batch_first=True, padding_value=0)
    tgt_batch = pad_sequence(tgt_batch, batch_first=True, padding_value=0)
    return src_batch, tgt_batch

train_src, val_src, train_tgt, val_tgt = train_test_split(
    urdu_texts, roman_texts, test_size=0.1, random_state=42
)
train_dataset = Seq2SeqDataset(train_src, train_tgt, sp_urdu, sp_roman)
val_dataset = Seq2SeqDataset(val_src, val_tgt, sp_urdu, sp_roman)
test_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)


# ============================
# Utilities
# ============================
def combine_bidir_hidden(h, c):
    h = torch.cat([h[0:h.size(0):2], h[1:h.size(0):2]], dim=2)
    c = torch.cat([c[0:c.size(0):2], c[1:c.size(0):2]], dim=2)
    return h, c

def cer(s1, s2):
    if s1 == s2:
        return 0.0
    return Levenshtein.distance(s1, s2) / max(len(s1), len(s2))

# ============================
# Luong Attention
# ============================
class LuongAttention(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.attn = nn.Linear(hidden_size*2, hidden_size*2)

    def forward(self, decoder_hidden, encoder_outputs, mask=None):
        energy = torch.bmm(encoder_outputs, decoder_hidden.unsqueeze(2)).squeeze(2)
        if mask is not None:
            energy = energy.masked_fill(mask == 0, -1e9)
        attn_weights = torch.softmax(energy, dim=1)
        context = torch.bmm(attn_weights.unsqueeze(1), encoder_outputs).squeeze(1)
        return context, attn_weights

# ============================
# Encoder & Decoder
# ============================
class Encoder(nn.Module):
    def __init__(self, input_size, emb_size=512, hidden_size=128, num_layers=2, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(input_size, emb_size)
        self.lstm = nn.LSTM(
            emb_size, hidden_size, num_layers=num_layers,
            batch_first=True, bidirectional=True, dropout=dropout
        )

    def forward(self, x):
        x = self.embedding(x)
        out, (h, c) = self.lstm(x)
        h, c = combine_bidir_hidden(h, c)
        return out, (h, c)

class DecoderWithAttention(nn.Module):
    def __init__(self, output_size, emb_size=512, hidden_size=128, num_layers=2, dropout=0.3): # Changed num_layers to 2
        super().__init__()
        self.embedding = nn.Embedding(output_size, emb_size)
        self.lstm = nn.LSTM(
            emb_size, hidden_size*2, num_layers=num_layers,
            batch_first=True, dropout=dropout
        )
        self.attention = LuongAttention(hidden_size)
        self.fc = nn.Linear(hidden_size*4, output_size)

    def forward(self, x, hidden, encoder_outputs, mask=None):
        x = self.embedding(x)
        out, hidden = self.lstm(x, hidden)
        outputs = []
        for t in range(out.size(1)):
            dec_h = out[:, t, :]
            context, _ = self.attention(dec_h, encoder_outputs, mask)
            concat = torch.cat([dec_h, context], dim=1)
            output = self.fc(concat)
            outputs.append(output.unsqueeze(1))
        outputs = torch.cat(outputs, dim=1)
        return outputs, hidden

# ============================
# Initialize Model
# ============================
encoder = Encoder(src_vocab_size).to(device)
decoder = DecoderWithAttention(tgt_vocab_size).to(device)
criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=0.001)
MAX_DECODING_LEN = 50


# ============================
# Evaluation Helpers
# ============================
def evaluate_accuracy(encoder, decoder, data_loader):
    encoder.eval()
    decoder.eval()
    total_correct, total_tokens = 0, 0
    with torch.no_grad():
        for src, tgt in data_loader:
            src, tgt = src.to(device), tgt.to(device)
            encoder_outputs, hidden = encoder(src)
            dec_input = tgt[:, 0].unsqueeze(1)
            outputs = []

            mask = src != 0
            for t in range(1, tgt.size(1)):
                dec_out, hidden = decoder(dec_input, hidden, encoder_outputs, mask)
                outputs.append(dec_out)
                dec_input = dec_out.argmax(2)
            outputs = torch.cat(outputs, dim=1)
            preds = outputs.argmax(2)
            mask_tgt = tgt[:, 1:] != 0
            total_correct += ((preds == tgt[:, 1:]) & mask_tgt).sum().item()
            total_tokens += mask_tgt.sum().item()
    return total_correct / total_tokens if total_tokens > 0 else 0

def evaluate_bleu(encoder, decoder, data_loader):
    encoder.eval()
    decoder.eval()
    total_bleu, count = 0, 0
    with torch.no_grad():
        for src, tgt in data_loader:
            src, tgt = src.to(device), tgt.to(device)
            encoder_outputs, hidden = encoder(src)
            dec_input = tgt[:, 0].unsqueeze(1)
            preds = []

            mask = src != 0
            for _ in range(tgt.size(1)-1):
                dec_out, hidden = decoder(dec_input, hidden, encoder_outputs, mask)
                dec_input = dec_out.argmax(2)
                preds.append(dec_input)
            preds = torch.cat(preds, dim=1).cpu().numpy()
            refs = tgt[:, 1:].cpu().numpy()
            for p, r in zip(preds, refs):
                p = [i for i in p if i != 0]
                r = [i for i in r if i != 0]
                if len(r) > 0:
                    total_bleu += sentence_bleu([r], p, smoothing_function=smooth_fn)
                    count += 1
    return total_bleu / count if count > 0 else 0

# ============================
# Training Loop
# ============================
EPOCHS = 15
teacher_forcing_ratio = 0.5
early_stopping = EarlyStopping(patience=5, mode="max")

for epoch in range(EPOCHS):
    encoder.train()
    decoder.train()
    total_loss, total_correct, total_tokens = 0, 0, 0

    for src, tgt in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        src, tgt = src.to(device), tgt.to(device)
        optimizer.zero_grad()
        encoder_outputs, hidden = encoder(src)
        dec_input = tgt[:, 0].unsqueeze(1)
        outputs = []

        mask = src != 0
        for t in range(1, tgt.size(1)):
            dec_out, hidden = decoder(dec_input, hidden, encoder_outputs, mask)
            outputs.append(dec_out)
            teacher_force = torch.rand(1).item() < teacher_forcing_ratio
            dec_input = tgt[:, t].unsqueeze(1) if teacher_force else dec_out.argmax(2)

        outputs = torch.cat(outputs, dim=1)
        loss = criterion(outputs.reshape(-1, tgt_vocab_size), tgt[:, 1:].reshape(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(list(encoder.parameters()) + list(decoder.parameters()), 1.0)
        optimizer.step()

        preds = outputs.argmax(2)
        mask_tgt = tgt[:, 1:] != 0
        total_correct += ((preds == tgt[:, 1:]) & mask_tgt).sum().item()
        total_tokens += mask_tgt.sum().item()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    ppl = math.exp(avg_loss)
    train_acc = total_correct / total_tokens

    if (epoch+1) % 5 == 0 or epoch == EPOCHS-1:
        val_acc = evaluate_accuracy(encoder, decoder, val_loader)
        val_bleu = evaluate_bleu(encoder, decoder, val_loader)
        print(f"Epoch {epoch+1}: Loss={avg_loss:.4f}, PPL={ppl:.2f}, "
              f"Train Acc={train_acc:.4f}, Val Acc={val_acc:.4f}, Val BLEU={val_bleu:.4f}")
        early_stopping(val_bleu, encoder, decoder)
        if early_stopping.early_stop:
            print(" Early stopping triggered. Restoring best model...")
            early_stopping.load_best_model(encoder, decoder)
            break
    else:
        print(f"Epoch {epoch+1}: Loss={avg_loss:.4f}, PPL={ppl:.2f}, Train Acc={train_acc:.4f}")

# ============================
# Final Evaluation
# ============================
train_acc = evaluate_accuracy(encoder, decoder, train_loader)
val_acc = evaluate_accuracy(encoder, decoder, val_loader)
test_acc = evaluate_accuracy(encoder, decoder, test_loader)
print(f"Training Accuracy={train_acc:.4f}")
print(f"Validation Accuracy={val_acc:.4f}")
print(f"Test Accuracy={test_acc:.4f}")
# ============================
# Inference / Test Function
# ============================
def translate_sentence(sentence, encoder, decoder, sp_src, sp_tgt, max_len=50):
    encoder.eval()
    decoder.eval()
    with torch.no_grad():
        # Preprocess input
        src_ids = sp_src.EncodeAsIds(preprocess_urdu(sentence))[:max_len]
        src_tensor = torch.tensor(src_ids).unsqueeze(0).to(device)
        encoder_outputs, hidden = encoder(src_tensor)
        dec_input = torch.tensor([sp_tgt.bos_id()]).unsqueeze(0).to(device)

        decoded_ids = []
        mask = src_tensor != 0
        prev_token = None  # track previous token to prevent repeats
        for _ in range(MAX_DECODING_LEN):
            dec_out, hidden = decoder(dec_input, hidden, encoder_outputs, mask)
            next_token = dec_out.argmax(2)

            # Stop if EOS or repeated token
            if next_token.item() == sp_tgt.eos_id() or next_token.item() == prev_token:
                break

            decoded_ids.append(next_token.item())
            dec_input = next_token
            prev_token = next_token.item()

        return sp_tgt.DecodeIds(decoded_ids)
# ============================
# Test Cases
# ============================
test_sentences = [
    "تجھ کو دیکھا ہے آج دیر کے بعد",
    "آج کا دن گزر نہ جائے کہیں"
]


print("\n Test Translations:")
for sent in test_sentences:
    pred = translate_sentence(sent, encoder, decoder, sp_urdu, sp_roman)
    print(f"Urdu: {sent}")
    print(f"Pred: {pred}\n")

Using device: cuda
 Line count mismatch in gar-na-andoh-e-shab-e-furqat-bayaan-ho-jaaegaa-mirza-ghalib-ghazals. Urdu=22, Roman=21
 Added 1 Roman lines to /content/dataset/dataset/mirza-ghalib/en/gar-na-andoh-e-shab-e-furqat-bayaan-ho-jaaegaa-mirza-ghalib-ghazals
 Line count mismatch in shumaar-e-subha-marguub-e-but-e-mushkil-pasand-aayaa-mirza-ghalib-ghazals-1. Urdu=14, Roman=13
 Added 1 Roman lines to /content/dataset/dataset/mirza-ghalib/en/shumaar-e-subha-marguub-e-but-e-mushkil-pasand-aayaa-mirza-ghalib-ghazals-1
 Line count mismatch in gar-tujh-ko-hai-yaqiin-e-ijaabat-duaa-na-maang-mirza-ghalib-ghazals. Urdu=18, Roman=17
 Added 1 Roman lines to /content/dataset/dataset/mirza-ghalib/en/gar-tujh-ko-hai-yaqiin-e-ijaabat-duaa-na-maang-mirza-ghalib-ghazals
 Line count mismatch in vaan-pahunch-kar-jo-gash-aataa-pae-ham-hai-ham-ko-mirza-ghalib-ghazals. Urdu=28, Roman=27
 Added 1 Roman lines to /content/dataset/dataset/mirza-ghalib/en/vaan-pahunch-kar-jo-gash-aataa-pae-ham-hai-ham-ko-mirz

Epoch 1: 100%|██████████| 593/593 [00:27<00:00, 21.39it/s]


Epoch 1: Loss=4.7926, PPL=120.62, Train Acc=0.3290


Epoch 2: 100%|██████████| 593/593 [00:26<00:00, 21.97it/s]


Epoch 2: Loss=2.8051, PPL=16.53, Train Acc=0.5713


...

Epoch 13: Loss=0.3135, PPL=1.37, Train Acc=0.9300


Epoch 14: 100%|██████████| 593/593 [00:27<00:00, 21.79it/s]


Epoch 14: Loss=0.2817, PPL=1.33, Train Acc=0.9372


Epoch 15: 100%|██████████| 593/593 [00:27<00:00, 21.69it/s]


Epoch 15: Loss=0.2588, PPL=1.30, Train Acc=0.9421, Val Acc=0.7122, Val BLEU=0.3918
Training Accuracy=0.9517
Validation Accuracy=0.7122
Test Accuracy=0.7122

 Test Translations:
Urdu: تجھ کو دیکھا ہے آج دیر کے بعد
Pred: tujh ko dekh hai aaj der ke baad

Urdu: آج کا دن گزر نہ جائے کہیں
Pred: aaj k din guzar na jaa.e kah



In [7]:
# Colab cell 1 — inspect files and session
import os, sys
print("Working dir:", os.getcwd())
print("\nFiles in /content:")
!ls -la /content

# check for sentencepiece files and checkpoint
for fname in ["sp_urdu.model","sp_roman.model","seq2seq_checkpoint.pt"]:
    print(fname, "=>", "FOUND" if os.path.exists(fname) else "MISSING")

# check python-session variables (if you trained in this session)
print("\nIn-memory model variables:")
print("encoder" in globals(), "decoder" in globals())


Working dir: /content

Files in /content:
total 5736
drwxr-xr-x 1 root root    4096 Sep 25 09:48 .
drwxr-xr-x 1 root root    4096 Sep 25 09:44 ..
drwxr-xr-x 4 root root    4096 Sep 23 13:39 .config
drwxr-xr-x 4 root root    4096 Sep 25 09:48 dataset
-rw-r--r-- 1 root root 2927519 Sep 25 09:47 dataset.zip
-rw-r--r-- 1 root root  743234 Sep 25 09:49 roman_corpus.txt
drwxr-xr-x 1 root root    4096 Sep 23 13:39 sample_data
-rw-r--r-- 1 root root  335093 Sep 25 09:49 sp_roman.model
-rw-r--r-- 1 root root  100391 Sep 25 09:49 sp_roman.vocab
-rw-r--r-- 1 root root  351723 Sep 25 09:49 sp_urdu.model
-rw-r--r-- 1 root root  116979 Sep 25 09:49 sp_urdu.vocab
-rw-r--r-- 1 root root 1267848 Sep 25 09:49 urdu_corpus.txt
sp_urdu.model => FOUND
sp_roman.model => FOUND
seq2seq_checkpoint.pt => MISSING

In-memory model variables:
True True


In [8]:
import torch, os
if 'encoder' in globals() and 'decoder' in globals():
    ckpt = {
        "encoder_state_dict": encoder.state_dict(),
        "decoder_state_dict": decoder.state_dict(),
        # include the config you trained with (adjust if different)
        "config": {"emb_size": 512, "hidden_size": 128, "num_layers": 2}
    }
    torch.save(ckpt, "seq2seq_checkpoint.pt")
    print("Saved seq2seq_checkpoint.pt in /content")
else:
    print("encoder/decoder NOT found in memory. If you haven't trained in this session, run your training cells first (or follow the training script you used earlier), then run this cell.")


Saved seq2seq_checkpoint.pt in /content


In [9]:
# Create requirements.txt file
requirements = """streamlit==1.28.0
torch==2.0.1
torchvision==0.15.2
torchaudio==2.0.2
sentencepiece==0.1.99
numpy==1.24.4
"""

with open("requirements.txt", "w") as f:
    f.write(requirements)

print("✅ requirements.txt created successfully!")


✅ requirements.txt created successfully!


In [11]:
!pip install -q streamlit pyngrok sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 105.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 149.3 MB/s eta 0:00:00


In [12]:
import streamlit as st
import torch
import torch.nn as nn
import sentencepiece as spm
import unicodedata
import re

# ------------------------------
# Preprocessing
# ------------------------------
def preprocess_urdu(text):
    text = unicodedata.normalize("NFC", text)
    text = re.sub(r"[\u064B-\u065F\u0670]", "", text)  # remove harakaat
    text = text.replace("۔", ".").replace("؟", "?").replace("،", ",")
    text = re.sub(r"\s+", " ", text).strip()
    return text

# ------------------------------
# Utilities
# ------------------------------
def combine_bidir_hidden(h, c):
    h = torch.cat([h[0:h.size(0):2], h[1:h.size(0):2]], dim=2)
    c = torch.cat([c[0:c.size(0):2], c[1:c.size(0):2]], dim=2)
    return h, c

# ------------------------------
# Luong Attention
# ------------------------------
class LuongAttention(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.attn = nn.Linear(hidden_size * 2, hidden_size * 2, bias=True)

    def forward(self, decoder_hidden, encoder_outputs, mask=None):
        energy = torch.bmm(encoder_outputs, decoder_hidden.unsqueeze(2)).squeeze(2)
        if mask is not None:
            energy = energy.masked_fill(mask == 0, -1e9)
        attn_weights = torch.softmax(energy, dim=1)
        context = torch.bmm(attn_weights.unsqueeze(1), encoder_outputs).squeeze(1)
        return context, attn_weights

# ------------------------------
# Encoder
# ------------------------------
class Encoder(nn.Module):
    def __init__(self, input_size, emb_size=512, hidden_size=128, num_layers=2, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(input_size, emb_size)
        self.lstm = nn.LSTM(emb_size, hidden_size, num_layers=num_layers,
                            batch_first=True, bidirectional=True, dropout=dropout)

    def forward(self, x):
        x = self.embedding(x)
        out, (h, c) = self.lstm(x)
        h, c = combine_bidir_hidden(h, c)
        return out, (h, c)

# ------------------------------
# Decoder with Attention
# ------------------------------
class DecoderWithAttention(nn.Module):
    def __init__(self, output_size, emb_size=512, hidden_size=128, num_layers=2, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(output_size, emb_size)
        self.lstm = nn.LSTM(emb_size, hidden_size*2, num_layers=num_layers,
                            batch_first=True, dropout=dropout)
        self.attention = LuongAttention(hidden_size)
        self.fc = nn.Linear(hidden_size*4, output_size)

    def forward(self, x, hidden, encoder_outputs, mask=None):
        x = self.embedding(x)
        out, hidden = self.lstm(x, hidden)
        outputs = []
        for t in range(out.size(1)):
            dec_h = out[:, t, :]
            context, _ = self.attention(dec_h, encoder_outputs, mask)
            concat = torch.cat([dec_h, context], dim=1)
            output = self.fc(concat)
            outputs.append(output.unsqueeze(1))
        outputs = torch.cat(outputs, dim=1)
        return outputs, hidden

# ------------------------------
# Load models & SentencePiece
# ------------------------------
@st.cache_resource
def load_models_local(ckpt_path='seq2seq_checkpoint.pt',
                      sp_urdu_path='sp_urdu.model',
                      sp_roman_path='sp_roman.model'):
    sp_urdu = spm.SentencePieceProcessor(model_file=sp_urdu_path)
    sp_roman = spm.SentencePieceProcessor(model_file=sp_roman_path)
    src_vocab = sp_urdu.get_piece_size()
    tgt_vocab = sp_roman.get_piece_size()

    checkpoint = torch.load(ckpt_path, map_location='cpu')
    cfg = checkpoint.get('config', {})
    emb_size = cfg.get('emb_size', 512)
    hidden_size = cfg.get('hidden_size', 128)
    num_layers = cfg.get('num_layers', 2)

    encoder = Encoder(src_vocab, emb_size=emb_size, hidden_size=hidden_size, num_layers=num_layers)
    decoder = DecoderWithAttention(tgt_vocab, emb_size=emb_size, hidden_size=hidden_size, num_layers=num_layers)

    encoder.load_state_dict(checkpoint['encoder_state_dict'], strict=False)
    decoder.load_state_dict(checkpoint['decoder_state_dict'], strict=False)
    encoder.eval(); decoder.eval()
    return encoder, decoder, sp_urdu, sp_roman

encoder, decoder, sp_urdu, sp_roman = load_models_local()
MAX_LEN = 500  # max input length

# ------------------------------
# Translate single chunk
# ------------------------------
def translate_text(sentence):
    enc_in = sp_urdu.EncodeAsIds(preprocess_urdu(sentence))[:MAX_LEN]
    if len(enc_in) == 0:
        return ""
    src_tensor = torch.tensor(enc_in).unsqueeze(0)
    encoder_outputs, hidden = encoder(src_tensor)
    dec_input = torch.tensor([sp_roman.bos_id()]).unsqueeze(0)
    decoded_ids = []
    mask = src_tensor != 0
    prev_token = None

    for _ in range(MAX_LEN):
        dec_out, hidden = decoder(dec_input, hidden, encoder_outputs, mask)
        next_token = dec_out.argmax(2)
        token = next_token.item()
        if token == sp_roman.eos_id() or token == prev_token:
            break
        decoded_ids.append(token)
        dec_input = next_token
        prev_token = token

    return sp_roman.DecodeIds(decoded_ids)

# ------------------------------
# Split long paragraph into chunks
# ------------------------------
def split_into_chunks(text, max_tokens=80):
    sentences = re.split(r'(?<=[.؟?!])\s+', text)
    chunks, buffer = [], []
    token_count = 0

    for s in sentences:
        if not s.strip():
            continue
        tokens = sp_urdu.EncodeAsIds(preprocess_urdu(s))
        if token_count + len(tokens) <= max_tokens:
            buffer.append(s)
            token_count += len(tokens)
        else:
            chunks.append(" ".join(buffer))
            buffer = [s]
            token_count = len(tokens)
    if buffer:
        chunks.append(" ".join(buffer))
    return chunks

# ------------------------------
# Translate multi-line text
# ------------------------------
def translate_block(text):
    """
    Translate multi-line Urdu text, sentence by sentence, preserving all lines.
    """
    lines = text.split("\n")
    translated_lines = []

    progress = st.progress(0)
    total_lines = len(lines)

    for idx, line in enumerate(lines):
        if not line.strip():
            translated_lines.append("")  # preserve blank lines
        else:
            # Split line into actual sentences first
            sentences = re.split(r'(?<=[.؟?!])\s+', line.strip())
            sentence_translations = [translate_text(s) for s in sentences if s.strip()]
            translated_lines.append(" ".join(sentence_translations))

        progress.progress((idx + 1) / total_lines)

    progress.empty()
    return translated_lines


# ------------------------------
# Streamlit UI
# ------------------------------
st.set_page_config(page_title="Urdu→Roman Transliteration", layout="centered")
st.title("Urdu → Roman Transliteration")

txt = st.text_area("Enter Urdu text (any number of sentences/paragraphs)", height=300)

if st.button("Translate"):
    if txt.strip():
        with st.spinner("Transliterating..."):
            translated_lines = translate_block(txt)
            for u, r in zip(txt.split("\n"), translated_lines):
                col1, col2 = st.columns(2)
                col1.markdown(f"**Urdu:** {u}")
                col2.markdown(f"**Roman:** {r}")



2025-09-25 09:59:37.338 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-25 09:59:37.406 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2025-09-25 09:59:37.407 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-25 09:59:37.408 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-25 09:59:37.409 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-25 09:59:37.578 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-25 09:59:37.579 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-25 09:59:37.579 Thread 'MainThread': mi

In [13]:
from pyngrok import ngrok

NGROK_AUTH_TOKEN = "33B6w5PZd9iKvOa1DNjYQkBBS9v_2f4Y4Xs5tfxALNWFaZw2E"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Kill any old ngrok sessions
ngrok.kill()

# Run Streamlit
get_ipython().system_raw("streamlit run app.py --server.port 8501 &")

# Expose URL
public_url = ngrok.connect(8501, "http")
print(" Your public URL:", public_url)


 Your public URL: NgrokTunnel: "https://gamogenetical-unjeweled-misha.ngrok-free.dev" -> "http://localhost:8501"
